In [ ]:
# ============================================================
# ERA5 Pressure Levels (0.25°) — India — parallel downloader
#
# Rewrite goals:
#   • Every variable pulls levels 250 / 500 / 850 together.
#   • VARIABLES is now just a list of names — add as many as
#     you want; each is downloaded as its own small request
#     (1 variable × 3 levels × half-year), which stays well
#     under the 120,000-field CDS limit and avoids the
#     "heavy request" queue penalty.
#   • GRIB download by default (native format, no slow
#     server-side netCDF conversion). Optional local convert.
#   • 6-hourly toggle (GraphCast/WeatherBench use 6h, not 1h)
#     — a 4× cut in fields and transfer if you don't need 1h.
#   • Workers sized to the CDS per-user concurrent limit;
#     more than that just fills your queue and buys nothing.
#   • Resume / validation / .part logic preserved as-is.
# ============================================================

import os
import time
import warnings
import threading
from datetime import date
from concurrent.futures import ThreadPoolExecutor, as_completed

import cdsapi
import xarray as xr

warnings.filterwarnings("ignore", category=xr.SerializationWarning)

# ------------------------------------------------------------
# Setup
# ------------------------------------------------------------

DOWNLOAD_DIR = r"downloads_era5_pressure"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

DATASET = "reanalysis-era5-pressure-levels"

# India (CDS format: [North, West, South, East])
AREA = [37.1, 68.12, 6.75, 97.42]

# Pressure levels applied to EVERY variable
PRESSURE_LEVELS = ["250", "500", "850"]

# ---- The only list you edit to add variables ----
# One request per variable (all 3 levels), so this can be long.
VARIABLES = [
    "u_component_of_wind",
    "v_component_of_wind",
    "temperature",
    "geopotential",
    "specific_humidity",
    "vertical_velocity",
    # add more here — relative_humidity, divergence, vorticity, ...
]

# ------------------------------------------------------------
# Format / temporal resolution
# ------------------------------------------------------------

# "grib" is native and faster on the new CDS (no server-side
# netCDF conversion). Reading grib needs cfgrib:  pip install cfgrib
# Flip to "netcdf" if your downstream must have .nc and you'd
# rather not convert locally.
DATA_FORMAT = "grib"          # "grib" (recommended) or "netcdf"

# Optional: after a grib download, also write a .nc alongside it.
# Requires cfgrib. Leave False if you consume grib directly.
CONVERT_TO_NETCDF = False

# Temporal resolution. Hourly = every hour. Set to 6 for the
# 00/06/12/18 subset used by GraphCast / WeatherBench (4× smaller).
TIMESTEP_HOURS = 1            # 1 or 6 (or any divisor of 24)

# Parallel downloads. The new CDS runs only a few of YOUR requests
# at once; extra workers just queue. 4–6 is the practical ceiling.
# Check current per-user limits: https://cds.climate.copernicus.eu/live
N_WORKERS = 4

# ------------------------------------------------------------
# Derived request pieces
# ------------------------------------------------------------

_EXT = "grib" if DATA_FORMAT == "grib" else "nc"
_ENGINE = "cfgrib" if DATA_FORMAT == "grib" else "netcdf4"

ALL_MONTHS = [f"{m:02d}" for m in range(1, 13)]
DAYS = [f"{d:02d}" for d in range(1, 32)]
TIMES = [f"{h:02d}:00" for h in range(0, 24, TIMESTEP_HOURS)]

# ------------------------------------------------------------
# Dynamic date range: Jan 1980 → (today - LAG_MONTHS)
# ------------------------------------------------------------

START_YEAR, START_MONTH = 1980, 1
LAG_MONTHS = 2

today = date.today()
end_y, end_m = today.year, today.month
for _ in range(LAG_MONTHS):
    end_m -= 1
    if end_m == 0:
        end_m = 12
        end_y -= 1

year_plan = []
for y in range(START_YEAR, end_y + 1):
    if y == START_YEAR and y == end_y:
        months = [f"{m:02d}" for m in range(START_MONTH, end_m + 1)]
    elif y == START_YEAR:
        months = [f"{m:02d}" for m in range(START_MONTH, 13)]
    elif y == end_y:
        months = [f"{m:02d}" for m in range(1, end_m + 1)]
    else:
        months = ALL_MONTHS
    year_plan.append((str(y), months))

# Split each year into Jan-Jun (H1) and Jul-Dec (H2)
H1_MONTHS = {f"{m:02d}" for m in range(1, 7)}
H2_MONTHS = {f"{m:02d}" for m in range(7, 13)}

period_plan = []
for year, months in year_plan:
    h1 = [m for m in months if m in H1_MONTHS]
    h2 = [m for m in months if m in H2_MONTHS]
    if h1:
        period_plan.append((year, "H1", h1))
    if h2:
        period_plan.append((year, "H2", h2))

# ------------------------------------------------------------
# Field-count sanity check (so you never trip the 120k limit)
# ------------------------------------------------------------

_max_days = 184  # worst-case half-year
_fields_per_req = len(PRESSURE_LEVELS) * _max_days * len(TIMES)

print(f"Today is          : {today}")
print(f"Year range        : {year_plan[0][0]} → {year_plan[-1][0]}")
print(f"Variables         : {len(VARIABLES)}")
print(f"Pressure levels   : {PRESSURE_LEVELS}")
print(f"Timestep          : {TIMESTEP_HOURS}h ({len(TIMES)} times/day)")
print(f"Format            : {DATA_FORMAT}")
print(f"Half-year periods : {len(period_plan)} per variable")
print(f"Total requests    : {len(period_plan) * len(VARIABLES)}")
print(f"Fields / request  : ~{_fields_per_req:,}  (CDS hourly limit = 120,000)")
if _fields_per_req > 120_000:
    print("  ⚠ Over the per-request field limit — switch batching to monthly.")
print(f"Parallel workers  : {N_WORKERS}")
print(f"Output dir        : {DOWNLOAD_DIR}")
print("Setup done.\n")

# Shared request template (levels + format live here)
BASE_EXTRAS = {
    "product_type": ["reanalysis"],
    "pressure_level": PRESSURE_LEVELS,
    "data_format": DATA_FORMAT,
    "download_format": "unarchived",
}

# ============================================================
# Helpers
# ============================================================

def is_valid_file(fpath):
    """Open with the right engine and confirm it holds data."""
    try:
        with xr.open_dataset(fpath, engine=_ENGINE) as d:
            dvs = list(d.data_vars)
            if not dvs:
                return False
            if d[dvs[0]].size == 0:
                return False
        return True
    except Exception:
        return False


def scan_existing(name, var_dir):
    """Resume support: drop .part files, keep valid, requeue corrupt."""
    needs = []
    n_valid = n_missing = n_corrupt = n_partial = 0

    for f in (os.listdir(var_dir) if os.path.isdir(var_dir) else []):
        if f.endswith(".part") or ".part." in f:
            try:
                os.remove(os.path.join(var_dir, f))
                n_partial += 1
            except OSError:
                pass

    for year, half, months in period_plan:
        outfile = os.path.join(var_dir, f"{name}_{year}_{half}.{_EXT}")
        if not os.path.exists(outfile):
            needs.append((year, half, months))
            n_missing += 1
        elif is_valid_file(outfile):
            n_valid += 1
        else:
            try:
                os.remove(outfile)
            except OSError:
                pass
            needs.append((year, half, months))
            n_corrupt += 1

    return needs, n_valid, n_missing, n_corrupt, n_partial


_print_lock = threading.Lock()


def safe_print(msg):
    with _print_lock:
        print(msg, flush=True)


def _maybe_convert(grib_path):
    """Optionally write a sibling .nc from a grib file."""
    if not (DATA_FORMAT == "grib" and CONVERT_TO_NETCDF):
        return
    nc_path = grib_path[:-len("grib")] + "nc"
    try:
        with xr.open_dataset(grib_path, engine="cfgrib") as d:
            d.to_netcdf(nc_path)
    except Exception as e:
        safe_print(f"      → netCDF convert failed for {os.path.basename(grib_path)}: {e}")


def download_one_period(name, year, half, months, var_dir):
    """Download one variable × 3 levels × half-year into a .part, then rename."""
    final_path = os.path.join(var_dir, f"{name}_{year}_{half}.{_EXT}")
    part_path = f"{final_path}.part.{threading.get_ident()}"

    request = {
        **BASE_EXTRAS,
        "variable": [name],
        "year": [year],
        "month": months,
        "day": DAYS,
        "time": TIMES,
        "area": AREA,
    }

    try:
        client = cdsapi.Client(quiet=True, wait_until_complete=True)
        client.retrieve(DATASET, request, part_path)
        os.replace(part_path, final_path)
        _maybe_convert(final_path)
        return (year, half, True, None)
    except Exception as e:
        if os.path.exists(part_path):
            try:
                os.remove(part_path)
            except OSError:
                pass
        return (year, half, False, str(e))


def download_variable(name):
    var_dir = os.path.join(DOWNLOAD_DIR, name)
    os.makedirs(var_dir, exist_ok=True)

    print("=" * 60)
    print(f"{name}   levels={PRESSURE_LEVELS}")
    print("=" * 60)

    t0 = time.time()
    print("  Scanning existing files...", end=" ", flush=True)
    needs, n_valid, n_missing, n_corrupt, n_partial = scan_existing(name, var_dir)

    parts = [f"{n_valid} valid"]
    if n_missing:
        parts.append(f"{n_missing} missing")
    if n_corrupt:
        parts.append(f"{n_corrupt} corrupted (deleted)")
    if n_partial:
        parts.append(f"{n_partial} partial (deleted)")
    print(", ".join(parts))

    if not needs:
        print(f"  → nothing to do  ({(time.time() - t0) / 60:.1f} min)\n")
        return 0

    print(f"  Downloading {len(needs)} half-year period(s) with {N_WORKERS} workers...")

    n_done = n_fail = completed = 0
    total = len(needs)

    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {
            pool.submit(download_one_period, name, year, half, months, var_dir):
                (year, half)
            for year, half, months in needs
        }
        for fut in as_completed(futures):
            year, half, ok, err = fut.result()
            completed += 1
            label = f"{year}-{half}"
            if ok:
                n_done += 1
                safe_print(f"    [{completed:>3}/{total}] {label}: done")
            else:
                n_fail += 1
                err_short = err[:200] + "..." if len(err) > 200 else err
                safe_print(f"    [{completed:>3}/{total}] {label}: FAILED: {err_short}")
                safe_print(f"      → if this is a volume/field-limit error, "
                           f"split {label} into monthly requests.")

    summary = f"  → {name}: {n_done} downloaded, {n_valid} pre-existing valid"
    if n_corrupt:
        summary += f", {n_corrupt} corrupted re-attempted"
    if n_fail:
        summary += f", {n_fail} FAILED"
    summary += f"  ({(time.time() - t0) / 60:.1f} min)"
    print(summary + "\n")

    return n_fail


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":
    overall_t0 = time.time()
    fail_summary = {name: download_variable(name) for name in VARIABLES}
    total_min = (time.time() - overall_t0) / 60.0

    print("=" * 60)
    print("ERA5 Pressure Levels (India) — COMPLETE")
    print("=" * 60)
    print(f"Total wall time: {total_min:.1f} min")

    if sum(fail_summary.values()):
        print(f"\n⚠ Failures by variable: {fail_summary}")
        print("  Re-run — already-valid files are skipped.")
    else:
        print("\n✓ No failures.")

In [ ]:
#arcos script

# ============================================================
# ERA5 → DAILY (mean / max / min) for India, from ARCO-ERA5
#
# Why this design:
#   You're unsure whether you need max/min or just mean. Instead
#   of committing to one, this streams HOURLY ERA5 from Google's
#   ARCO store (no CDS queue) and computes daily mean, max, AND
#   min in one pass — so you keep every statistic. The hourly data
#   never lands on disk; only the small daily files are written.
#   Precipitation is accumulated, so it gets a daily SUM.
#
# Data: gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3
#       (the GraphCast/NeuralGCM superset; surface + 37 pressure levels)
#
# Deps: pip install xarray zarr gcsfs dask netCDF4 numpy
#
# NOTE: run over a couple of test years first (set END_YEAR small)
# to confirm variable names and volumes before the full pull.
# ============================================================

import os
import numpy as np
import xarray as xr

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

ARCO_PATH = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"
OUT_DIR = "era5_daily_india"
os.makedirs(OUT_DIR, exist_ok=True)

# India box. ARCO latitude runs 90 → -90 (descending), so the
# latitude slice is (North, South). Longitude is 0 → 359.75; India
# stays positive so no 0/360 wrap needed.
LAT_N, LAT_S = 37.1, 6.75
LON_W, LON_E = 68.12, 97.42

LEVELS = [250, 500, 850]

START_YEAR = 1980
END_YEAR = 2024          # start small (e.g. 1981) to test, then widen

# ---- Variable groups (ARCO analysis-ready names) ----
# Pressure-level, instantaneous → daily mean/max/min at LEVELS
PRESSURE_VARS = [
    "u_component_of_wind",
    "v_component_of_wind",
    "temperature",
    "specific_humidity",
]
# Relative humidity may not exist in the AR store. If missing it is
# derived from specific_humidity + temperature + pressure (see below).
PRESSURE_VARS_OPTIONAL = ["relative_humidity"]

# Surface, instantaneous → daily mean/max/min
# (the "at surface" analogues of your u/v/temp/humidity, plus MSLP)
SURFACE_INSTANT_VARS = [
    "mean_sea_level_pressure",
    "2m_temperature",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "2m_dewpoint_temperature",     # surface humidity proxy; drop if unwanted
]
# Surface, accumulated → daily SUM
SURFACE_ACCUM_VARS = ["total_precipitation"]

INSTANT_STATS = {"mean": "mean", "max": "max", "min": "min"}

DERIVE_RH_IF_MISSING = True

# ------------------------------------------------------------
# Open store (lazy)
# ------------------------------------------------------------

ds = xr.open_zarr(ARCO_PATH, chunks={"time": 24 * 31},
                  storage_options={"token": "anon"})

# Clamp to the data actually available in the store
avail_start = np.datetime64(ds.attrs["valid_time_start"])
avail_stop = np.datetime64(ds.attrs["valid_time_stop"])

present = set(ds.data_vars)

def keep(names):
    ok, missing = [v for v in names if v in present], [v for v in names if v not in present]
    for m in missing:
        print(f"  ⚠ not in store, skipping: {m}")
    return ok

pl_vars = keep(PRESSURE_VARS)
pl_opt = keep(PRESSURE_VARS_OPTIONAL)
sfc_inst = keep(SURFACE_INSTANT_VARS)
sfc_acc = keep(SURFACE_ACCUM_VARS)

need_rh = ("relative_humidity" in PRESSURE_VARS_OPTIONAL
           and "relative_humidity" not in pl_opt
           and DERIVE_RH_IF_MISSING)

print(f"ARCO availability : {avail_start} → {avail_stop}")
print(f"Pressure vars     : {pl_vars + pl_opt}"
      f"{'  (+ derived relative_humidity)' if need_rh else ''}")
print(f"Surface instant   : {sfc_inst}")
print(f"Surface accum(sum): {sfc_acc}")
print(f"Levels            : {LEVELS}")
print(f"Years             : {START_YEAR}–{END_YEAR}\n")

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def derive_rh(temp_K, q, level_hPa):
    """Relative humidity (%) from specific humidity, temperature, pressure.
    Bolton (1980) saturation vapour pressure. level_hPa broadcasts over 'level'."""
    e = q * level_hPa / (0.622 + 0.378 * q)                 # vapour pressure (hPa)
    es = 6.112 * np.exp(17.67 * (temp_K - 273.15) / (temp_K - 29.65))
    return (100.0 * e / es).clip(0, 100)


def is_valid(path):
    try:
        with xr.open_dataset(path) as d:
            return len(d.data_vars) > 0 and d[list(d.data_vars)[0]].size > 0
    except Exception:
        return False


def daily_stats_for_month(month_ds, accum_vars):
    """Return a dataset of daily stats for one month, already in memory."""
    g = month_ds.resample(time="1D")
    out = {}
    for v in month_ds.data_vars:
        if v in accum_vars:
            out[f"{v}_sum"] = g.sum()[v]
        else:
            for suffix, how in INSTANT_STATS.items():
                out[f"{v}_{suffix}"] = getattr(g, how)()[v]
    return xr.Dataset(out)


def process_year(year):
    out_path = os.path.join(OUT_DIR, f"era5_daily_india_{year}.nc")
    if os.path.exists(out_path):
        if is_valid(out_path):
            print(f"  {year}: already done, skipping")
            return
        os.remove(out_path)

    y0 = np.datetime64(f"{year}-01-01T00")
    y1 = np.datetime64(f"{year}-12-31T23")
    if y1 < avail_start or y0 > avail_stop:
        print(f"  {year}: outside ARCO range, skipping")
        return

    monthly = []
    for m in range(1, 13):
        t0 = np.datetime64(f"{year}-{m:02d}-01T00")
        t1 = (np.datetime64(f"{year}-{m+1:02d}-01T00") - np.timedelta64(1, "h")
              if m < 12 else np.datetime64(f"{year}-12-31T23"))
        if t1 < avail_start or t0 > avail_stop:
            continue

        # Pressure-level view (subset levels), surface view (no level dim)
        pl = ds[pl_vars + pl_opt].sel(
            level=LEVELS,
            latitude=slice(LAT_N, LAT_S),
            longitude=slice(LON_W, LON_E),
            time=slice(t0, t1),
        )
        sfc = ds[sfc_inst + sfc_acc].sel(
            latitude=slice(LAT_N, LAT_S),
            longitude=slice(LON_W, LON_E),
            time=slice(t0, t1),
        )

        # Pull this month into memory once (bounded ~<1 GB for India box)
        pl = pl.load()
        sfc = sfc.load()

        if need_rh and "specific_humidity" in pl and "temperature" in pl:
            pl["relative_humidity"] = derive_rh(
                pl["temperature"], pl["specific_humidity"], pl["level"]
            )

        pl_daily = daily_stats_for_month(pl, accum_vars=[])
        sfc_daily = daily_stats_for_month(sfc, accum_vars=sfc_acc)
        monthly.append(xr.merge([pl_daily, sfc_daily]))
        print(f"    {year}-{m:02d} done")

    if not monthly:
        print(f"  {year}: no data in range")
        return

    year_ds = xr.concat(monthly, dim="time")
    comp = {v: {"zlib": True, "complevel": 4} for v in year_ds.data_vars}
    year_ds.to_netcdf(out_path, encoding=comp)
    print(f"  {year}: written → {out_path}\n")


# ------------------------------------------------------------
# Main
# ------------------------------------------------------------

if __name__ == "__main__":
    for year in range(START_YEAR, END_YEAR + 1):
        try:
            process_year(year)
        except Exception as e:
            print(f"  {year}: FAILED — {e}\n")
    print("Done. Re-run to fill any years that failed (existing years are skipped).")

# cds script


In [ ]:
# ============================================================
# ERA5 DAILY statistics — CDS derived datasets — India
# For S2S. Fastest CDS path to DAILY data: the derived
# daily-statistics datasets aggregate server-side and return
# tiny files (24× less transfer than downloading hourly).
#
# Datasets (published late 2024):
#   derived-era5-pressure-levels-daily-statistics  (u,v,T,q,RH @ levels)
#   derived-era5-single-levels-daily-statistics    (precip, MSLP, 2m, 10m)
#
# Design choices (see notes at each config block):
#   • daily_mean for state variables, daily_sum for precipitation.
#     To also grab extremes, add "daily_maximum"/"daily_minimum"
#     to STATISTICS — costs one extra request per variable-month.
#   • One variable per request (dataset requirement), per MONTH —
#     CDS's per-request "cost limit" (based on underlying hourly
#     data touched, not output size) rejects year-sized requests
#     for this area/variable combo with a 403; monthly batches
#     stay comfortably under it.
#   • 6-hourly sampling for means (fast); 1-hourly forced for
#     sums and extremes (needed for correctness / true extremes).
#   • Derived requests are server-compute heavy → modest worker count.
#   • Output is a zip per request; the .nc inside is extracted.
#   • Resume: valid files skipped, corrupt/partial redone.
#   • Files land one-per-month; combine with xarray.open_mfdataset
#     per variable once downloaded (see bottom of file for a snippet).
#
# Deps: pip install "cdsapi>=0.7" xarray netCDF4
# ============================================================

import os
import time
import random
import shutil
import zipfile
import warnings
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date

import cdsapi
import xarray as xr

warnings.filterwarnings("ignore", category=xr.SerializationWarning)

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

DOWNLOAD_DIR = r"downloads_era5_daily"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# India (CDS: [North, West, South, East])
AREA = [37.1, 68.12, 6.75, 97.42]

PRESSURE_LEVELS = ["250", "500", "850"]

START_YEAR = 1980
LAG_MONTHS = 2                      # skip most recent (ERA5 production lag)

# ---- STATISTIC CHOICE (the main speed lever) ----
# Speed-optimal default: mean only (precip always uses sum below).
# To hedge for extremes, add the max/min values — one extra request
# per variable-year each. If a max/min request 400s, the exact enum
# may be "daily_max"/"daily_min"; grab it from the dataset's
# "Show API request" button on the CDS site.
STATISTICS = ["daily_mean"]        # e.g. + ["daily_maximum", "daily_minimum"]

TIME_ZONE = "utc+00:00"

# The new CDS caps how many jobs one user can have IN THE QUEUE at
# once. Exceed it and new submissions are rejected outright (400,
# "the job has been rejected, number of queued requests..."). This
# cap is low and dynamic for the derived datasets, so keep workers
# small; the retry logic below absorbs the occasional rejection by
# waiting and resubmitting. Check the live value at
# https://cds.climate.copernicus.eu/live and nudge up only if you
# see no rejections.
N_WORKERS = 1

# Transient failures (queue-full rejection, cost/queue limits, 5xx,
# timeouts) are retried with exponential backoff; permanent ones
# (bad keyword, invalid request) fail fast.
MAX_RETRIES = 6

PL_DATASET = "derived-era5-pressure-levels-daily-statistics"
SL_DATASET = "derived-era5-single-levels-daily-statistics"

# ---- Variables (11) ----
# family: "pl" (pressure levels) or "sfc"; accum=True → daily_sum only.
VARIABLES = [
    # pressure-level, instantaneous
    # ("u_component_of_wind",      "pl",  False),
    ("v_component_of_wind",      "pl",  False),
    ("temperature",              "pl",  False),
    ("specific_humidity",        "pl",  False),
    ("relative_humidity",        "pl",  False),
    # surface, instantaneous
    ("mean_sea_level_pressure",  "sfc", False),
    ("2m_temperature",           "sfc", False),
    ("10m_u_component_of_wind",  "sfc", False),
    ("10m_v_component_of_wind",  "sfc", False),
    ("2m_dewpoint_temperature",  "sfc", False),
    # surface, accumulated → daily sum
    ("total_precipitation",      "sfc", True),
]

# ------------------------------------------------------------
# Date range -> per-MONTH batches
#
# The new CDS enforces a per-request "cost limit" based on how
# much underlying hourly data the request touches, not on the
# (tiny) daily output size. A full year, one variable, India-box
# request can exceed it even though the result is a few KB.
# Batching by month keeps each request's cost trivial. This is
# 12x more requests than the year-batch version, but each is
# near-instant to queue and process, so wall-clock impact is
# small -- and it is the reliable fix for the 403 cost error.
# ------------------------------------------------------------

today = date.today()
end_y, end_m = today.year, today.month
for _ in range(LAG_MONTHS):
    end_m -= 1
    if end_m == 0:
        end_m = 12
        end_y -= 1

ALL_MONTHS = [f"{m:02d}" for m in range(1, 13)]
DAYS = [f"{d:02d}" for d in range(1, 32)]

year_plan = []  # (year, [single_month]) pairs -- name kept for minimal diff
for y in range(START_YEAR, end_y + 1):
    last_m = end_m if y == end_y else 12
    for m in range(1, last_m + 1):
        year_plan.append((str(y), [f"{m:02d}"]))


def stats_for(accum):
    """Which (statistic, frequency) pairs to fetch for a variable."""
    if accum:
        return [("daily_sum", "1_hourly")]          # sum needs full 24 h
    out = []
    for s in STATISTICS:
        freq = "6_hourly" if s == "daily_mean" else "1_hourly"  # extremes need 1 h
        out.append((s, freq))
    return out


# Build the full task list
tasks = []
for name, family, accum in VARIABLES:
    for stat, freq in stats_for(accum):
        for year, months in year_plan:
            tasks.append((name, family, accum, stat, freq, year, months))

print(f"Today is        : {today}")
print(f"Year range      : {year_plan[0][0]} → {year_plan[-1][0]}")
print(f"Variables       : {len(VARIABLES)}  |  PL levels {PRESSURE_LEVELS}")
print(f"Statistics      : {STATISTICS}  (+ daily_sum for precip)")
print(f"Total requests  : {len(tasks)}")
print(f"Parallel workers: {N_WORKERS}")
print(f"Output dir      : {DOWNLOAD_DIR}")
print("Setup done.\n")

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def is_valid(path):
    try:
        with xr.open_dataset(path, engine="netcdf4") as d:
            return bool(d.data_vars) and d[list(d.data_vars)[0]].size > 0
    except Exception:
        return False


def out_path_for(name, stat, year, month):
    var_dir = os.path.join(DOWNLOAD_DIR, name)
    os.makedirs(var_dir, exist_ok=True)
    return os.path.join(var_dir, f"{name}_{stat}_{year}_{month}.nc")


def finalize(tmp, final):
    """Derived daily returns a zip with one .nc inside; extract it."""
    if zipfile.is_zipfile(tmp):
        with zipfile.ZipFile(tmp) as z:
            ncs = [n for n in z.namelist() if n.endswith(".nc")]
            if not ncs:
                raise RuntimeError("zip contained no .nc")
            with z.open(ncs[0]) as src, open(final, "wb") as dst:
                shutil.copyfileobj(src, dst)
        os.remove(tmp)
    else:
        os.replace(tmp, final)


_print_lock = threading.Lock()


def safe_print(msg):
    with _print_lock:
        print(msg, flush=True)


def _is_transient(msg):
    m = (msg or "").lower()
    signals = ("rejected", "queued", "number of queued", "too many",
               "cost limit", "429", "500", "502", "503", "504",
               "timeout", "timed out", "temporarily", "connection",
               "max retries", "service unavailable")
    return any(s in m for s in signals)


def download_task(task):
    name, family, accum, stat, freq, year, months = task
    month = months[0]
    final = out_path_for(name, stat, year, month)
    label = f"{name}/{stat}/{year}-{month}"

    if os.path.exists(final):
        if is_valid(final):
            return (label, "skip", None)
        os.remove(final)

    tmp = f"{final}.part.{threading.get_ident()}"
    dataset = PL_DATASET if family == "pl" else SL_DATASET

    request = {
        "product_type": "reanalysis",
        "variable": [name],
        "year": [year],
        "month": months,
        "day": DAYS,
        "daily_statistic": stat,
        "time_zone": TIME_ZONE,
        "frequency": freq,
        "area": AREA,
        "data_format": "netcdf",
    }
    if family == "pl":
        request["pressure_level"] = PRESSURE_LEVELS

    last_err = None
    for attempt in range(MAX_RETRIES):
        try:
            client = cdsapi.Client(quiet=True, wait_until_complete=True, retry_max=1)
            client.retrieve(dataset, request, tmp)
            finalize(tmp, final)
            if not is_valid(final):
                raise RuntimeError("downloaded file failed validation")
            return (label, "done", None)
        except Exception as e:
            last_err = str(e)
            for p in (tmp, final):
                if os.path.exists(p):
                    try:
                        os.remove(p)
                    except OSError:
                        pass
            # Only wait-and-retry on transient conditions (queue full, etc.)
            if attempt < MAX_RETRIES - 1 and _is_transient(last_err):
                wait = min(300, 15 * (2 ** attempt)) + random.uniform(0, 8)
                safe_print(f"      {label}: queue busy, retry in {wait:.0f}s")
                time.sleep(wait)
                continue
            break

    return (label, "fail", last_err)


# ------------------------------------------------------------
# Main
# ------------------------------------------------------------

if __name__ == "__main__":
    t0 = time.time()

    # Pre-scan so the progress counter reflects real work
    todo = [t for t in tasks if not is_valid(out_path_for(t[0], t[3], t[5], t[6][0]))]
    n_skip = len(tasks) - len(todo)
    print(f"{n_skip} already valid, {len(todo)} to download.\n")

    done = fail = 0
    fails = []
    completed = 0
    total = len(todo)

    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {pool.submit(download_task, t): t for t in todo}
        for fut in as_completed(futures):
            label, status, err = fut.result()
            completed += 1
            if status == "done":
                done += 1
                safe_print(f"  [{completed:>4}/{total}] {label}: done")
            else:
                fail += 1
                fails.append(label)
                short = err[:180] + "..." if err and len(err) > 180 else err
                safe_print(f"  [{completed:>4}/{total}] {label}: FAILED: {short}")

    mins = (time.time() - t0) / 60.0
    print("\n" + "=" * 60)
    print(f"DONE in {mins:.1f} min — {done} downloaded, {n_skip} pre-existing, {fail} failed")
    print("=" * 60)
    if fails:
        print("Failed tasks (re-run to retry; valid files are skipped):")
        for f in fails[:40]:
            print(f"  {f}")
        if len(fails) > 40:
            print(f"  ... and {len(fails) - 40} more")

# ------------------------------------------------------------
# After downloading: files are one-per-variable-per-month. To get
# a single time series per variable, concatenate with xarray, e.g.:
#
#   import xarray as xr
#   ds = xr.open_mfdataset(
#       "downloads_era5_daily/temperature/temperature_daily_mean_*.nc",
#       combine="by_coords",
#   )
#   ds.to_netcdf("temperature_daily_mean_full.nc")
#
# Run this per variable (and per statistic, for precip's daily_sum).
# ------------------------------------------------------------

Today is        : 2026-07-05
Year range      : 1980 → 2026
Variables       : 10  |  PL levels ['250', '500', '850']
Statistics      : ['daily_mean']  (+ daily_sum for precip)
Total requests  : 5570
Parallel workers: 1
Output dir      : downloads_era5_daily
Setup done.

0 already valid, 5570 to download.

      v_component_of_wind/daily_mean/1980-01: queue busy, retry in 21s
      v_component_of_wind/daily_mean/1980-01: queue busy, retry in 32s
      v_component_of_wind/daily_mean/1980-01: queue busy, retry in 66s
      v_component_of_wind/daily_mean/1980-01: queue busy, retry in 127s
      v_component_of_wind/daily_mean/1980-01: queue busy, retry in 245s
  [   1/5570] v_component_of_wind/daily_mean/1980-01: FAILED: 400 Client Error: Bad Request for url: https://cds.climate.copernicus.eu/api/retrieve/v1/jobs/89e9b5a8-e0f9-4133-9418-19d1dd298838/results
The job has been rejected
Number queued r...
      v_component_of_wind/daily_mean/1980-02: queue busy, retry in 21s
      v_component_of

In [ ]:
# ============================================================
# ERA5 daily data for India — SIMPLE serial version
#
# One request at a time. No threads. Because only ONE job is ever
# in the CDS queue, you never hit the per-user queue limit, so
# there is nothing to thrash on — it just works through the list.
#
# Downloads daily MEAN for state variables and daily SUM for
# precipitation, from the CDS derived daily-statistics datasets
# (server aggregates; you download small files, not hourly bulk).
#
# DEFAULT SCOPE IS SMALL so you get real files today. To pull the
# full record, set START_YEAR = 1980 and leave it running — the
# derived-daily queue makes that a multi-hour/overnight job. That
# slowness is the CDS platform, not this script.
#
# Deps: pip install cdsapi xarray netCDF4
# ============================================================

import os
import time
import shutil
import zipfile
from datetime import date

import cdsapi

OUT_DIR = "era5_daily"
os.makedirs(OUT_DIR, exist_ok=True)

AREA = [37.1, 68.12, 6.75, 97.42]     # India: N, W, S, E
LEVELS = ["250", "500", "850"]
DAYS = [f"{d:02d}" for d in range(1, 32)]

# ---- SCOPE (edit these two) ----
# Quick test = last 2 full years. Full record = set START_YEAR = 1980.
LAG_MONTHS = 2
_t = date.today()
_ey, _em = _t.year, _t.month
for _ in range(LAG_MONTHS):
    _em -= 1
    if _em == 0:
        _em, _ey = 12, _ey - 1
END_YEAR = _ey
START_YEAR = END_YEAR - 1              # <-- change to 1980 for everything

# ---- Variables ----
# (cds_name, family, accumulated?)  family: "pl" or "sfc"
VARIABLES = [
    # ("u_component_of_wind",      "pl",  False),
    ("v_component_of_wind",      "pl",  False),
    ("temperature",              "pl",  False),
    ("specific_humidity",        "pl",  False),
    ("relative_humidity",        "pl",  False),
    ("mean_sea_level_pressure",  "sfc", False),
    ("2m_temperature",           "sfc", False),
    ("10m_u_component_of_wind",  "sfc", False),
    ("10m_v_component_of_wind",  "sfc", False),
    ("2m_dewpoint_temperature",  "sfc", False),
    ("total_precipitation",      "sfc", True),   # daily SUM
]

PL_DS = "derived-era5-pressure-levels-daily-statistics"
SL_DS = "derived-era5-single-levels-daily-statistics"

client = cdsapi.Client(quiet=True, wait_until_complete=True)


def unzip_to(tmp, final):
    """Derived daily returns a zip holding one .nc — pull it out."""
    if zipfile.is_zipfile(tmp):
        with zipfile.ZipFile(tmp) as z:
            nc = [n for n in z.namelist() if n.endswith(".nc")][0]
            with z.open(nc) as s, open(final, "wb") as d:
                shutil.copyfileobj(s, d)
        os.remove(tmp)
    else:
        os.replace(tmp, final)


def fetch(name, family, accum, year, month):
    var_dir = os.path.join(OUT_DIR, name)
    os.makedirs(var_dir, exist_ok=True)
    final = os.path.join(var_dir, f"{name}_{year}_{month}.nc")
    if os.path.exists(final) and os.path.getsize(final) > 0:
        return "skip"

    if accum:
        stat, freq = "daily_sum", "1_hourly"      # sum needs full 24 h
    else:
        stat, freq = "daily_mean", "6_hourly"     # mean: 6-hourly is fine & cheap

    req = {
        "product_type": "reanalysis",
        "variable": [name],
        "year": [year],
        "month": [month],
        "day": DAYS,
        "daily_statistic": stat,
        "time_zone": "utc+00:00",
        "frequency": freq,
        "area": AREA,
        "data_format": "netcdf",
    }
    if family == "pl":
        req["pressure_level"] = LEVELS

    dataset = PL_DS if family == "pl" else SL_DS
    tmp = final + ".part"

    for attempt in range(3):                       # tiny retry for hiccups
        try:
            client.retrieve(dataset, req, tmp)
            unzip_to(tmp, final)
            return "done"
        except Exception as e:
            if os.path.exists(tmp):
                try:
                    os.remove(tmp)
                except OSError:
                    pass
            if attempt < 2:
                time.sleep(60)
            else:
                return f"FAIL: {str(e)[:120]}"


# ------------------------------------------------------------
# Run: plain nested loop, one request at a time
# ------------------------------------------------------------

months = [f"{m:02d}" for m in range(1, 13)]
jobs = []
for name, family, accum in VARIABLES:
    for year in range(START_YEAR, END_YEAR + 1):
        last_m = _em if year == END_YEAR else 12
        for m in range(1, last_m + 1):
            jobs.append((name, family, accum, str(year), f"{m:02d}"))

print(f"Scope: {START_YEAR}-{END_YEAR}  |  {len(VARIABLES)} vars  |  {len(jobs)} requests")
print("Running one request at a time (no queue thrashing).\n")

t0 = time.time()
done = skip = fail = 0
for i, job in enumerate(jobs, 1):
    name, family, accum, year, month = job
    result = fetch(name, family, accum, year, month)
    if result == "done":
        done += 1
    elif result == "skip":
        skip += 1
    else:
        fail += 1
    mins = (time.time() - t0) / 60
    print(f"[{i:>4}/{len(jobs)}] {name} {year}-{month}: {result}   "
          f"({done} done, {skip} skip, {fail} fail, {mins:.0f} min)")

print(f"\nFinished: {done} downloaded, {skip} already had, {fail} failed.")
if fail:
    print("Re-run to retry failures — finished files are skipped.")

Scope: 2025-2026  |  10 vars  |  170 requests
Running one request at a time (no queue thrashing).

[   1/170] v_component_of_wind 2025-01: FAIL: 400 Client Error: Bad Request for url: https://cds.climate.copernicus.eu/api/retrieve/v1/jobs/5a82a030-8be3-42e5-8220-58   (0 done, 0 skip, 1 fail, 2 min)
[   2/170] v_component_of_wind 2025-02: FAIL: 400 Client Error: Bad Request for url: https://cds.climate.copernicus.eu/api/retrieve/v1/jobs/a2bbbe59-aa39-4085-893b-d4   (0 done, 0 skip, 2 fail, 5 min)
[   3/170] v_component_of_wind 2025-03: FAIL: 400 Client Error: Bad Request for url: https://cds.climate.copernicus.eu/api/retrieve/v1/jobs/3dedd71e-b85e-45a2-ac7c-62   (0 done, 0 skip, 3 fail, 8 min)
[   4/170] v_component_of_wind 2025-04: FAIL: 400 Client Error: Bad Request for url: https://cds.climate.copernicus.eu/api/retrieve/v1/jobs/aba59389-a720-4c51-a8cc-ee   (0 done, 0 skip, 4 fail, 10 min)
[   5/170] v_component_of_wind 2025-05: FAIL: 400 Client Error: Bad Request for url: https://cds